In [ ]:
import numpy as np
# parameter for FDM
DT = 0.01
NSKIP = 4
NXD = 2
DX = 0.4 *NXD
NX = 800 // NXD

X0 = 40.0

bddep = np.full(NX+1, 0.0)
XB0 = 150.0
XB1 = 180.0
Bth = 4.0
for i in range(NX+1):
    if i*DX >= XB0 and i*DX < XB1:
        bddep[i] = Bth/(XB1-XB0)*(i*DX-XB0)
    elif i*DX >= XB1:
        bddep[i] = Bth


In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import glob
import os

# 1. 設定
folder_path = './'  # ファイルがあるディレクトリ
file_pattern = 'wav.l.*.dat'
files = sorted(glob.glob(os.path.join(folder_path, file_pattern)))
fact = 3e4

sampling_rate = 1/(DT*NSKIP)
F1 = 0.05
F2 = 0.52
order = 2

nyquist = 0.5 * sampling_rate
low  = F1 / nyquist
high = F2 / nyquist

plt.figure(figsize=(6, 4))

# 2. データの読み込みとプロット
for file in files:
    # ファイル名から距離(km)を抽出 (例: wav.h.0010.dat -> 10)
    filename = os.path.basename(file)
    distance_km = int(filename.split('.')[2]) 
    
    try:
        data = np.loadtxt(file)
        time = data[:, 0]
        vz = data[:, 2]*fact
        vx = data[:, 1]*fact
        # 0 : time
        # 1 : Vx
        # 2 : Vz
        sos = signal.butter(order, [low, high], btype='band', output='sos')
        vz_filt = signal.sosfiltfilt(sos, vz)
        #vx_filt = signal.sosfiltfilt(sos, vx)

        offset = distance_km - X0
        plt.plot(vz_filt + offset, time, color='black')
        #plt.plot(time, vx_filt + offset, color='red')
        
    except Exception as e:
        print(f"Error reading {filename}: {e}")

# 3. グラフの装飾
plt.ylabel('Time [s]')
plt.xlabel('Vz + Offset (Distance) [km]')
plt.grid(True, linestyle='--', alpha=0.5)
# データ数が多い場合は凡例を外に出すか、非表示に
# plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left') 
plt.xlim(-40,280)
plt.ylim(0,150)
plt.tight_layout()
plt.show()

In [ ]:
file1 = 'wav.l.0210.dat'
file0 = 'output/wav.l.0210.dat'

data = np.loadtxt(file1)
time1 = data[:, 0]
vz1 = data[:, 2]

F1 = 0.05
F2 = 0.5

sampling_rate = 1.0/(time1[1]-time1[0])
nyquist = 0.5 * sampling_rate
low  = F1 / nyquist
high = F2 / nyquist
sos = signal.butter(order, [low, high], btype='band', output='sos')
vz1_filt = signal.sosfiltfilt(sos, vz1)

data = np.loadtxt(file0)
time0 = data[:, 0]
vz0 = data[:, 2]

sampling_rate = 1.0/(time0[1]-time0[0])
nyquist = 0.5 * sampling_rate
low  = F1 / nyquist
high = F2 / nyquist
sos = signal.butter(order, [low, high], btype='band', output='sos')
vz0_filt = signal.sosfiltfilt(sos, vz0)

plt.figure(figsize=(10, 3))

plt.plot(time0, vz0_filt, color='gray')
plt.plot(time1, vz1_filt, color='blue')
plt.xlim(0,150)
plt.xlabel('Time [s]',fontsize=14)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# --- 設定 ---
fact = 1e4 # scale factor

time_val = 62.0
step = int(time_val/DT)
step = int(step/100)*100
time_val = step*DT
filename = f"psv.l.{step:05d}.out"
   

fig, axs = plt.subplots(2, 2, figsize=(15, 8))
plt.rcParams.update({'font.size': 10})

# データ整形
out = pd.read_csv(filename, sep='\s+', names=('X','Z','VX','VZ','div','rot'))

NX = len(np.unique(out.X))
NZ = len(np.unique(out.Z))

X  = out["X"].values.reshape(NZ,NX)
Z  = out["Z"].values.reshape(NZ,NX)
VX = out["VX"].values.reshape(NZ,NX)
VZ = out["VZ"].values.reshape(NZ,NX)
VP = out["div"].values.reshape(NZ,NX)
VS = out["rot"].values.reshape(NZ,NX)
    
# 正規化
VX *= fact
VZ *= fact
VP *= fact*4
VS *= fact

# 左側：Vx
ax0 = axs[0,0]
im1 = ax0.pcolormesh(X, Z, VX, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$V_X$ at {time_val:.2f} s"
ax0.text(0.95, 0.05, label_text,
        transform=ax0.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax0.set_ylabel('Z (Depth) [km]',fontsize=14)

# 右側：Vz
ax1 = axs[0,1]
im2 = ax1.pcolormesh(X, Z, VZ, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$V_Z$ at {time_val:.2f} s"
ax1.text(0.95, 0.05, label_text,
        transform=ax1.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')

# 左側：div v
ax0 = axs[1,0]
im1 = ax0.pcolormesh(X, Z, VP, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$\mathrm{{div}} \ \mathbf{{v}} \times 4$ at {time_val:.2f} s"
ax0.text(0.95, 0.05, label_text,
        transform=ax0.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax0.set_xlabel('X [km]',fontsize=14)
ax0.set_ylabel('Z (Depth)[km]',fontsize=14)

# 右側：rot v
ax1 = axs[1,1]
im2 = ax1.pcolormesh(X, Z, VS, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$\mathrm{{rot}} \ \mathbf{{v}}$ at {time_val:.2f} s"
ax1.text(0.95, 0.05, label_text,
        transform=ax1.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax1.set_xlabel('X [km]',fontsize=14)

xb = np.arange(-0.2,320.2, DX)
for ax in axs.flatten():
    ax.set_xlim(0 , 320)
    ax.set_ylim(90, -10)
    ax.set_aspect(1.0)
    ax.plot(xb,np.full_like(xb,0),ls='-',color='black')
    ax.plot(xb,np.full_like(xb,3),ls='--',color='black')
    ax.plot(xb,np.full_like(xb,18),ls='--',color='black')
    ax.plot(xb,np.full_like(xb,33),ls='--',color='black')
    ax.plot(xb,bddep,ls='-',color='black',lw=2)
    ax.plot(xb,bddep/2,ls='-',color='black',lw=1)
    ax.tick_params(direction="in", top=True, right=True, which='both')

plt.tight_layout()
plt.show()